# 02 — Métricas e Figuras para SBCARS 2026

Análise comparativa **MAS Pipeline vs. Baseline** para os dois tamanhos de amostra:
- `n=100` (seed=42) — experimentos preliminares
- `n=625` — dataset completo

**Checklist SBCARS:**
- [ ] Tabela de métricas por modelo × idioma × arquitetura
- [ ] F1-macro: pipeline vs baseline (n=100 e n=625)
- [ ] MCC: pipeline vs baseline
- [ ] Teste de Wilcoxon bilateral (SQ2)
- [ ] Bootstrap CI 10.000 reamostras (SQ2)
- [ ] Cliff's delta — tamanho do efeito
- [ ] Figuras exportadas em alta resolução (300 dpi)

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from evaluation.metrics.statistical import pairwise_report, bootstrap_ci, wilcoxon_test, cliffs_delta

FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

SUMMARY = ROOT / "experiments/results/grid_summary_full.csv"

df = pd.read_csv(SUMMARY)
df = df[df["status"] == "ok"].copy()
df["model_short"] = df["model"].str.replace("ollama/", "").str.replace("+.*", "", regex=True)

print(f"Runs carregados: {len(df)}")
print(f"n disponíveis  : {sorted(df['n'].dropna().unique().tolist())}")
print(df.groupby(["strategy", "n"])["run_id"].count().rename("runs").to_string())

## 1. Tabela completa de métricas (n=100 e n=625)

In [ ]:
for n_filter in [100.0, 625.0]:
    subset = df[df["n"] == n_filter].copy()
    if subset.empty:
        print(f"Sem dados para n={int(n_filter)}")
        continue

    pivot = subset.pivot_table(
        index=["model_short", "lang"],
        columns="strategy",
        values=["accuracy", "f1_macro", "mcc"],
        aggfunc="mean",
    ).round(3)
    pivot.columns = [f"{m}_{s}" for m, s in pivot.columns]
    pivot = pivot.sort_index()

    print(f"\n{'='*70}")
    print(f"  n = {int(n_filter)}")
    print(f"{'='*70}")
    display(pivot)

## 2. F1-macro e MCC — Pipeline vs Baseline por modelo

In [ ]:
def plot_metric_comparison(df_subset, metric, n_label, ax):
    models  = sorted(df_subset["model_short"].unique())
    x       = np.arange(len(models))
    width   = 0.35

    baseline_vals = [
        df_subset[(df_subset["model_short"] == m) & (df_subset["strategy"] == "baseline")][metric].mean()
        for m in models
    ]
    pipeline_vals = [
        df_subset[(df_subset["model_short"] == m) & (df_subset["strategy"] == "pipeline")][metric].mean()
        for m in models
    ]

    ax.bar(x - width/2, baseline_vals, width, label="Baseline", color="#4C72B0", alpha=0.85)
    ax.bar(x + width/2, pipeline_vals, width, label="Pipeline (MAS)", color="#55A868", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} — {n_label}")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col, n_filter in enumerate([100.0, 625.0]):
    subset = df[df["n"] == n_filter]
    n_label = f"n={int(n_filter)}"
    for row, metric in enumerate(["f1_macro", "mcc"]):
        plot_metric_comparison(subset, metric, n_label, axes[row][col])

plt.suptitle("Pipeline MAS vs. Baseline — PROMISE NFR", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pipeline_vs_baseline_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Testes Estatísticos — SQ2 (Wilcoxon + Bootstrap CI + Cliff's delta)

In [ ]:
rows = []

for n_filter in [100.0, 625.0]:
    subset = df[df["n"] == n_filter]
    for metric in ["f1_macro", "mcc", "accuracy"]:
        pipeline_vals = subset[subset["strategy"] == "pipeline"][metric].dropna().tolist()
        baseline_vals = subset[subset["strategy"] == "baseline"][metric].dropna().tolist()

        # Alinha pares (mesmo modelo + lang)
        paired = subset.pivot_table(
            index=["model_short", "lang"], columns="strategy", values=metric
        ).dropna()
        if paired.empty or len(paired) < 2:
            continue

        a = paired["pipeline"].tolist()
        b = paired["baseline"].tolist()

        report = pairwise_report(metric, a, b, seed=42)

        rows.append({
            "n":          int(n_filter),
            "metric":     metric,
            "mean_pipeline": round(report.mean_a, 3),
            "mean_baseline": round(report.mean_b, 3),
            "Δ mean":     round(report.mean_a - report.mean_b, 3),
            "CI lower":   report.bootstrap_ci.ci_lower,
            "CI upper":   report.bootstrap_ci.ci_upper,
            "p (Wilcoxon)": round(report.wilcoxon.p_value, 4),
            "significant": "✓" if report.wilcoxon.significant else "✗",
            "Cliff's δ":  report.cliffs_delta.delta,
            "magnitude":  report.cliffs_delta.magnitude,
        })

results_df = pd.DataFrame(rows).set_index(["n", "metric"])
display(results_df)

## 4. Bootstrap CI — visualização do intervalo de confiança

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, n_filter in zip(axes, [100.0, 625.0]):
    subset_rows = results_df.loc[int(n_filter)] if int(n_filter) in results_df.index.get_level_values("n") else pd.DataFrame()
    if subset_rows.empty:
        ax.set_title(f"n={int(n_filter)} — sem dados")
        continue

    metrics = subset_rows.index.tolist()
    means   = subset_rows["Δ mean"].tolist()
    lowers  = subset_rows["CI lower"].tolist()
    uppers  = subset_rows["CI upper"].tolist()
    yerr    = [[m - l for m, l in zip(means, lowers)],
               [u - m for m, u in zip(means, uppers)]]

    colors = ["#55A868" if m > 0 else "#C44E52" for m in means]
    ax.barh(metrics, means, xerr=yerr, color=colors, alpha=0.8,
            error_kw={"elinewidth": 2, "capsize": 5, "ecolor": "black"})
    ax.axvline(0, color="gray", linewidth=1, linestyle="--")
    ax.set_xlabel("Δ médio (Pipeline − Baseline)")
    ax.set_title(f"Bootstrap CI 95% — n={int(n_filter)}")
    ax.xaxis.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Diferença média Pipeline vs. Baseline (10.000 reamostras)", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bootstrap_ci.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. F1-macro por idioma (PT vs EN)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, n_filter in zip(axes, [100.0, 625.0]):
    subset = df[df["n"] == n_filter]
    lang_strategy = subset.groupby(["lang", "strategy"])["f1_macro"].mean().unstack()

    x = np.arange(len(lang_strategy))
    width = 0.35
    ax.bar(x - width/2, lang_strategy.get("baseline", [0]*len(x)), width, label="Baseline", color="#4C72B0", alpha=0.85)
    ax.bar(x + width/2, lang_strategy.get("pipeline", [0]*len(x)), width, label="Pipeline (MAS)", color="#55A868", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(lang_strategy.index)
    ax.set_ylabel("F1-macro médio")
    ax.set_title(f"F1-macro por Idioma — n={int(n_filter)}")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Impacto do Idioma (PT vs EN) no F1-macro", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "f1_by_language.png", dpi=300, bbox_inches="tight")
plt.show()